<a href="https://colab.research.google.com/github/ArjiNasim/market-risk-pnl-attribution/blob/main/Market_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration visuelle pour les graphiques potentiels
sns.set_theme(style="whitegrid")
pd.set_option('display.float_format', lambda x: '%.2f' % x)
np.random.seed(42) # Pour la reproductibilité des données synthétiques

In [2]:
def generate_synthetic_data(date_str):
    desks = ['FIC_Rates', 'FIC_Credit', 'EQD_Delta', 'EQD_Vol']
    n_records = 500

    # Données de Risques et Sensibilités
    df_risk = pd.DataFrame({
        'Date': pd.to_datetime(date_str),
        'Trade_ID': [f'TRD_{i:04d}' for i in range(n_records)],
        'Desk': np.random.choice(desks, n_records),
        'Delta': np.random.normal(0, 50000, n_records),
        'Gamma': np.random.normal(0, 10000, n_records),
        'Vega': np.random.normal(1000, 20000, n_records),
        'DV01': np.random.normal(0, 5000, n_records),
        'VaR_Hist_99': np.random.uniform(10000, 150000, n_records),
        'VaR_Param_99': np.random.uniform(9000, 160000, n_records),
        'Stress_Test_Global': np.random.uniform(50000, 500000, n_records)
    })

    # Données de P&L Attribution
    df_pnl = pd.DataFrame({
        'Date': pd.to_datetime(date_str),
        'Trade_ID': df_risk['Trade_ID'],
        'Desk': df_risk['Desk'],
        'PnL_Market': np.random.normal(5000, 20000, n_records),
        'PnL_Carry': np.random.normal(1000, 2000, n_records),
        'PnL_Unexplained': np.random.normal(0, 3000, n_records)
    })
    df_pnl['Daily_PnL'] = df_pnl['PnL_Market'] + df_pnl['PnL_Carry'] + df_pnl['PnL_Unexplained']

    # Référentiel de Limites par Desk
    df_limits = pd.DataFrame({
        'Desk': desks,
        'Limit_VaR_99': [2000000, 2500000, 3000000, 4000000],
        'Limit_Stress_Test': [8000000, 9000000, 12000000, 15000000],
        'Limit_DV01': [150000, 200000, 0, 0], # EQD n'est pas géré par le DV01 principalement
        'Limit_Vega': [0, 50000, 500000, 2000000]
    })

    return df_risk, df_pnl, df_limits

# Exécution de la génération pour la journée d'aujourd'hui
target_date = datetime.date.today().strftime('%Y-%m-%d')
df_risk, df_pnl, df_limits = generate_synthetic_data(target_date)
print(f"Génération terminée : {len(df_risk)} trades générés pour le {target_date}.")

Génération terminée : 500 trades générés pour le 2026-11-11.


In [3]:
def aggregate_and_certify(df_risk, df_pnl):
    # Jointure des bases Risque et P&L
    df_master = pd.merge(df_risk, df_pnl, on=['Date', 'Trade_ID', 'Desk'])

    # Mapping des classes d'actifs (Asset Class)
    df_master['Asset_Class'] = df_master['Desk'].apply(lambda x: 'FIC' if 'FIC' in x else 'EQD')

    # Agrégation par Desk
    metrics_to_sum = ['Delta', 'Gamma', 'Vega', 'DV01', 'VaR_Hist_99', 'VaR_Param_99',
                      'Stress_Test_Global', 'PnL_Market', 'PnL_Carry', 'PnL_Unexplained', 'Daily_PnL']

    df_agg_desk = df_master.groupby(['Asset_Class', 'Desk'])[metrics_to_sum].sum().reset_index()

    # Contrôle de cohérence (Certification)
    # Règle 1: La VaR Paramétrique et Historique ne doivent pas diverger de plus de 30%
    df_agg_desk['VaR_Divergence'] = abs(df_agg_desk['VaR_Hist_99'] - df_agg_desk['VaR_Param_99']) / df_agg_desk['VaR_Hist_99']
    df_agg_desk['Certification_Status'] = np.where(df_agg_desk['VaR_Divergence'] > 0.30, 'WARNING: VaR Divergence', 'PASSED')

    return df_master, df_agg_desk

df_master, df_agg_desk = aggregate_and_certify(df_risk, df_pnl)
display(df_agg_desk[['Desk', 'VaR_Hist_99', 'VaR_Param_99', 'Certification_Status']])

,Desk,VaR_Hist_99,VaR_Param_99,Certification_Status
0,EQD_Delta,9873461.88,10728038.80,PASSED
1,EQD_Vol,11141606.77,12259501.25,PASSED
2,FIC_Credit,8930633.33,8927850.99,PASSED
3,FIC_Rates,10188560.63,10468780.18,PASSED


In [4]:
def monitor_limits(df_agg_desk, df_limits):
    # Jointure avec le référentiel des limites
    df_monitoring = pd.merge(df_agg_desk, df_limits, on='Desk')

    alerts = []

    for index, row in df_monitoring.iterrows():
        desk = row['Desk']

        # Contrôle VaR
        if row['VaR_Hist_99'] > row['Limit_VaR_99']:
            alerts.append({'Desk': desk, 'Metric': 'VaR_Hist_99', 'Value': row['VaR_Hist_99'],
                           'Limit': row['Limit_VaR_99'], 'Severity': 'CRITICAL'})

        # Contrôle Stress Test
        if row['Stress_Test_Global'] > row['Limit_Stress_Test']:
            alerts.append({'Desk': desk, 'Metric': 'Stress_Test', 'Value': row['Stress_Test_Global'],
                           'Limit': row['Limit_Stress_Test'], 'Severity': 'HIGH'})

        # Contrôle Vega
        if row['Limit_Vega'] > 0 and abs(row['Vega']) > row['Limit_Vega']:
            alerts.append({'Desk': desk, 'Metric': 'Vega', 'Value': abs(row['Vega']),
                           'Limit': row['Limit_Vega'], 'Severity': 'MEDIUM'})

    df_alerts = pd.DataFrame(alerts)
    return df_monitoring, df_alerts

df_monitoring, df_alerts = monitor_limits(df_agg_desk, df_limits)

if not df_alerts.empty:
    print(f"⚠️ {len(df_alerts)} brèches de limites détectées !")
    display(df_alerts)
else:
    print("✅ Aucune brèche de limite détectée aujourd'hui.")

⚠️ 9 brèches de limites détectées !


,Desk,Metric,Value,Limit,Severity
0,EQD_Delta,VaR_Hist_99,9873461.88,3000000,CRITICAL
1,EQD_Delta,Stress_Test,33079706.42,12000000,HIGH
2,EQD_Vol,VaR_Hist_99,11141606.77,4000000,CRITICAL
3,EQD_Vol,Stress_Test,37480713.15,15000000,HIGH
4,FIC_Credit,VaR_Hist_99,8930633.33,2500000,CRITICAL
5,FIC_Credit,Stress_Test,30776287.11,9000000,HIGH
6,FIC_Credit,Vega,469087.32,50000,MEDIUM
7,FIC_Rates,VaR_Hist_99,10188560.63,2000000,CRITICAL
8,FIC_Rates,Stress_Test,32911572.03,8000000,HIGH


In [5]:
def analyze_pnl_attribution(df_agg_desk, threshold_ratio=0.15, threshold_abs=50000):
    df_pnl_analysis = df_agg_desk[['Asset_Class', 'Desk', 'Daily_PnL', 'PnL_Market', 'PnL_Carry', 'PnL_Unexplained']].copy()

    # Calcul du ratio d'inexpliqué (en valeur absolue par rapport au P&L total absolu)
    df_pnl_analysis['Unexplained_Ratio'] = abs(df_pnl_analysis['PnL_Unexplained']) / (abs(df_pnl_analysis['Daily_PnL']) + 1)

    # Flagging des anomalies
    # Condition : Le P&L inexpliqué dépasse x% du P&L total OU dépasse un montant absolu critique
    condition_ratio = df_pnl_analysis['Unexplained_Ratio'] > threshold_ratio
    condition_abs = abs(df_pnl_analysis['PnL_Unexplained']) > threshold_abs

    df_pnl_analysis['Pnl_Quality_Flag'] = np.where(condition_ratio | condition_abs, 'INVESTIGATION REQUIRED', 'OK')

    return df_pnl_analysis

df_pnl_analysis = analyze_pnl_attribution(df_agg_desk)
display(df_pnl_analysis[['Desk', 'Daily_PnL', 'PnL_Unexplained', 'Unexplained_Ratio', 'Pnl_Quality_Flag']])

,Desk,Daily_PnL,PnL_Unexplained,Unexplained_Ratio,Pnl_Quality_Flag
0,EQD_Delta,713981.61,-1764.80,0.00,OK
1,EQD_Vol,663952.22,-43938.68,0.07,OK
2,FIC_Credit,803061.15,6332.93,0.01,OK
3,FIC_Rates,563582.46,-19231.61,0.03,OK


In [6]:
def generate_executive_summary(df_monitoring, df_alerts, df_pnl_analysis, date_str):
    total_pnl = df_pnl_analysis['Daily_PnL'].sum()
    total_var = df_monitoring['VaR_Hist_99'].sum()

    summary = f"### Executive Risk & P&L Summary - {date_str} ###\n\n"
    summary += f"1. Performance Globale :\n"
    summary += f"Le P&L global de la salle s'établit à {total_pnl:,.0f} EUR.\n"

    # Analyse des investigations P&L
    desks_investigation = df_pnl_analysis[df_pnl_analysis['Pnl_Quality_Flag'] == 'INVESTIGATION REQUIRED']['Desk'].tolist()
    if desks_investigation:
        summary += f"⚠️ Attention : L'Unexplained P&L nécessite une investigation sur les desks suivants : {', '.join(desks_investigation)}.\n\n"
    else:
        summary += f"✅ L'Income Attribution est de bonne qualité sur l'ensemble des périmètres.\n\n"

    summary += f"2. Consommation des Risques :\n"
    summary += f"La VaR Historique globale agrégée (somme arithmétique simple) est de {total_var:,.0f} EUR.\n"

    if not df_alerts.empty:
        summary += f"⚠️ {len(df_alerts)} dépassements de limites ont été détectés, dont {len(df_alerts[df_alerts['Severity']=='CRITICAL'])} critiques.\n"
        for _, alert in df_alerts.iterrows():
            summary += f"   - Desk {alert['Desk']} : {alert['Metric']} atteint {alert['Value']:,.0f} (Limite: {alert['Limit']:,.0f})\n"
    else:
        summary += "✅ Aucune brèche de limite recensée ce jour.\n"

    return summary

def export_data_for_dashboarding(df_master, df_monitoring, df_alerts, date_str):
    # Création d'un dossier d'export
    os.makedirs('exports', exist_ok=True)

    # Export en Parquet (très performant pour l'intégration PowerBI/Streamlit)
    df_master.to_parquet(f'exports/trade_level_data_{date_str}.parquet', index=False)
    df_monitoring.to_parquet(f'exports/desk_level_metrics_{date_str}.parquet', index=False)
    df_alerts.to_csv(f'exports/active_alerts_{date_str}.csv', index=False) # CSV pour lecture rapide

    print("Les données ont été exportées avec succès dans le dossier /exports/ (Formats Parquet & CSV).")

# Exécution finale
executive_summary = generate_executive_summary(df_monitoring, df_alerts, df_pnl_analysis, target_date)
print(executive_summary)

export_data_for_dashboarding(df_master, df_monitoring, df_alerts, target_date)

### Executive Risk & P&L Summary - 2026-11-11 ###

1. Performance Globale :
Le P&L global de la salle s'établit à 2,744,577 EUR.
✅ L'Income Attribution est de bonne qualité sur l'ensemble des périmètres.

2. Consommation des Risques :
La VaR Historique globale agrégée (somme arithmétique simple) est de 40,134,263 EUR.
⚠️ 9 dépassements de limites ont été détectés, dont 4 critiques.
   - Desk EQD_Delta : VaR_Hist_99 atteint 9,873,462 (Limite: 3,000,000)
   - Desk EQD_Delta : Stress_Test atteint 33,079,706 (Limite: 12,000,000)
   - Desk EQD_Vol : VaR_Hist_99 atteint 11,141,607 (Limite: 4,000,000)
   - Desk EQD_Vol : Stress_Test atteint 37,480,713 (Limite: 15,000,000)
   - Desk FIC_Credit : VaR_Hist_99 atteint 8,930,633 (Limite: 2,500,000)
   - Desk FIC_Credit : Stress_Test atteint 30,776,287 (Limite: 9,000,000)
   - Desk FIC_Credit : Vega atteint 469,087 (Limite: 50,000)
   - Desk FIC_Rates : VaR_Hist_99 atteint 10,188,561 (Limite: 2,000,000)
   - Desk FIC_Rates : Stress_Test atteint 32,